# Workload model training and export

This notebook trains a scikit-learn workload forecast model from Supabase worklogs and exports a model artifact used by `!workload` in Slack.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

workspace_root = Path.cwd()
if workspace_root.name.lower() == "ml":
    workspace_root = workspace_root.parent

python_dir = workspace_root / "backend" / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import workload_model as workload_model_module
importlib.reload(workload_model_module)

aggregate_monthly = workload_model_module.aggregate_monthly
forecast_with_artifact = workload_model_module.forecast_with_artifact
prepare_weekly_data = workload_model_module.prepare_weekly_data
resolve_model_path = workload_model_module.resolve_model_path
save_model_artifact = workload_model_module.save_model_artifact
train_best_model = workload_model_module.train_best_model

env_path = workspace_root / "backend" / "src" / "config" / ".env"
load_dotenv(env_path)

supabase_url = os.getenv("SUPABASE_URL")
supabase_service_key = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
if not supabase_url or not supabase_service_key:
    raise ValueError("SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY must be set in backend/src/config/.env")

supabase = create_client(supabase_url, supabase_service_key)
print(f"Using env file: {env_path}")
print(f"Python dir: {python_dir}")

Using env file: c:\Users\Albin\Desktop\Repos\Kruse\backend\src\config\.env
Python dir: c:\Users\Albin\Desktop\Repos\Kruse\backend\python


In [19]:
def fetch_all_worklogs(page_size=1000):
    rows = []
    offset = 0

    while True:
        end = offset + page_size - 1
        response = (
            supabase
            .table("WORKLOGS")
            .select("time_spent_seconds,started_at,user_id")
            .order("started_at", desc=False)
            .range(offset, end)
            .execute()
        )

        batch = response.data or []
        rows.extend(batch)

        if len(batch) < page_size:
            break

        offset += page_size

    return rows

worklogs = fetch_all_worklogs()
print(f"Fetched worklogs: {len(worklogs)}")

weekly_df = prepare_weekly_data(worklogs)
monthly_df = aggregate_monthly(weekly_df)

print(f"Weekly samples: {len(weekly_df)}")
print(f"Monthly samples: {len(monthly_df)}")
display(monthly_df.tail(12))

Fetched worklogs: 24330
Weekly samples: 140
Monthly samples: 32


c:\Users\Albin\Desktop\Repos\Kruse\backend\python\workload_model.py:52: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  
c:\Users\Albin\Desktop\Repos\Kruse\backend\python\workload_model.py:72: UserWarning: Converting to Period representation will drop timezone information.
  


,month_period,y,active_users,month
20,2025-04,4344.050000,21.80,2025-04
21,2025-05,3790.800000,21.75,2025-05
22,2025-06,3180.650000,20.00,2025-06
23,2025-07,2358.950000,14.40,2025-07
24,2025-08,2212.416667,17.50,2025-08
25,2025-09,2865.600000,17.60,2025-09
26,2025-10,2436.100000,19.50,2025-10
27,2025-11,2632.566667,19.50,2025-11
28,2025-12,2360.483333,17.20,2025-12
29,2026-01,2436.500000,19.75,2026-01


In [ ]:
artifact = train_best_model(monthly_df)

print("Selected model:", artifact["model_name"])
print("Trained at:", artifact["trained_at"])
print("Feature columns:", artifact["feature_columns"])
print("Validation type:", artifact.get("validation_type", "n/a"))
print("Candidates:")

candidates_df = pd.DataFrame(artifact["candidates"])
valid_df = candidates_df[candidates_df["rmse"].notna()].copy()
failed_df = candidates_df[candidates_df["rmse"].isna()].copy()

if len(valid_df) == 0:
    raise ValueError("No valid model candidates were trained. Check candidate errors below.")

valid_df = valid_df.sort_values(["rmse", "mae"]).reset_index(drop=True)
display(valid_df)

if len(failed_df) > 0:
    print("\nFailed candidates:")
    display(failed_df[["model", "error"]])

print("\nTop 3 by RMSE:")
display(valid_df.head(3))

C:\Users\Albin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.327e+06, tolerance: 1.062e+03
  model = cd_fast.enet_coordinate_descent(
C:\Users\Albin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.558e+06, tolerance: 1.084e+03
  model = cd_fast.enet_coordinate_descent(
C:\Users\Albin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\loca

Selected model: Ridge_a10
Trained at: 2026-04-20T08:43:52.026022
Feature columns: ['lag1', 'lag2', 'lag3', 'lag4', 'lag_mean_2', 'lag_mean_4', 'lag_std_4', 'lag_diff_1', 'trend', 'sin12', 'cos12']
Validation type: walk_forward
Candidates:


,model,mae,rmse,mape,folds
0,Ridge_a10,463.0021,571.4322,16.3408,14
1,ExtraTrees_d5,426.2220,604.4125,16.2359,14
2,Huber,518.7449,605.6416,18.2864,14
3,ExtraTrees_d3,429.8234,623.1472,16.6968,14
4,RandomForest_d5,459.4936,666.2777,17.8880,14
5,RandomForest_d3,453.9448,667.0742,17.7806,14
6,Ridge_a1,522.3589,725.5670,17.5484,14
7,GradientBoosting,548.7785,806.8354,21.3453,14
8,ElasticNet_a0.05_l1_0.5,638.3745,946.6073,20.2888,14
9,HistGradientBoosting,874.7348,975.8361,33.3417,14



Top 3 by RMSE:


,model,mae,rmse,mape,folds
0,Ridge_a10,463.0021,571.4322,16.3408,14
1,ExtraTrees_d5,426.2220,604.4125,16.2359,14
2,Huber,518.7449,605.6416,18.2864,14


In [21]:
model_path = resolve_model_path()
save_model_artifact(artifact, model_path)
print(f"Saved model artifact to: {model_path}")

Saved model artifact to: C:\Users\Albin\Desktop\Repos\Kruse\backend\python\models\workload_forecast_model.joblib


In [22]:
preview = forecast_with_artifact(monthly_df, artifact, periods_months=3)
preview_df = pd.DataFrame(preview)
display(preview_df)

,month,predicted_hours,lower_bound,upper_bound
0,2026-05,2644.39,1546.85,3741.93
1,2026-06,2376.63,1279.09,3474.17
2,2026-07,2170.51,1072.97,3268.05
